# Document Extraction with Docling

Docling is an open-source document parsing library by IBM that converts PDFs, Word documents, PowerPoint files, HTML, images, and more into structured, AI-ready formats.

**Supported formats:** PDF, DOCX, PPTX, XLSX, HTML, Images (PNG/JPG), AsciiDoc, Markdown

## 1. Installation

In [ ]:
# Install docling and optional dependencies
%pip install docling
%pip install docling-core

## 2. Imports & Setup

In [ ]:
import json
import time
from pathlib import Path

from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TesseractCliOcrOptions,
)
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)

print("Docling imported successfully!")

## 3. Basic Document Conversion (URL or File Path)

Convert any document — PDF, DOCX, HTML — with a single call.

In [ ]:
# Initialize the converter
converter = DocumentConverter()

# ----- Option A: convert from a URL -----
source_url = "https://arxiv.org/pdf/2408.09869"  # Docling paper itself

start = time.time()
result = converter.convert(source_url)
elapsed = time.time() - start

doc = result.document
print(f"Converted in {elapsed:.1f}s")
print(f"Number of pages : {len(doc.pages) if hasattr(doc, 'pages') else 'N/A'}")
print(f"Document type   : {type(doc).__name__}")

In [ ]:
# ----- Option B: convert a local file -----
# Uncomment and set your file path
# local_path = Path("./sample.pdf")
# result = converter.convert(local_path)
# doc = result.document

## 4. Export to Markdown

In [ ]:
markdown_output = doc.export_to_markdown()

# Preview first 2000 characters
print(markdown_output[:2000])
print("\n... (truncated)")

# Save to file
Path("output.md").write_text(markdown_output, encoding="utf-8")
print("\nSaved to output.md")

## 5. Export to JSON (structured)

In [ ]:
doc_dict = doc.export_to_dict()

# Save full JSON
with open("output.json", "w", encoding="utf-8") as f:
    json.dump(doc_dict, f, indent=2, ensure_ascii=False)

# Preview top-level keys
print("Top-level keys in JSON output:")
for key in doc_dict.keys():
    print(f"  - {key}")

print("\nSaved to output.json")

## 6. Table Extraction

In [ ]:
import pandas as pd
from docling_core.types.doc import TableItem

tables = [item for item, _ in doc.iterate_items() if isinstance(item, TableItem)]
print(f"Found {len(tables)} table(s) in the document.\n")

for idx, table in enumerate(tables):
    print(f"--- Table {idx + 1} ---")
    try:
        df = table.export_to_dataframe()
        print(df.to_string(index=False))
    except Exception as e:
        print(f"Could not render as DataFrame: {e}")
        # Fall back to markdown
        print(table.export_to_markdown())
    print()

## 7. Iterate Over Document Elements

Access headings, paragraphs, lists, figures, tables individually.

In [ ]:
from docling_core.types.doc import (
    TextItem,
    SectionHeaderItem,
    ListItem,
    PictureItem,
    TableItem,
)

element_counts = {}

for item, _ in doc.iterate_items():
    label = type(item).__name__
    element_counts[label] = element_counts.get(label, 0) + 1

print("Element type counts:")
for name, count in sorted(element_counts.items(), key=lambda x: -x[1]):
    print(f"  {name:30s}: {count}")

In [ ]:
# Print all section headings
print("Section Headings:\n" + "-" * 40)
for item, _ in doc.iterate_items():
    if isinstance(item, SectionHeaderItem):
        indent = "  " * (item.level - 1) if hasattr(item, "level") else ""
        print(f"{indent}{item.text}")

## 8. PDF-Specific Options: OCR, Table Detection

In [ ]:
# Enable OCR + advanced table detection for scanned PDFs
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True                # run OCR on image-based PDFs
pipeline_options.do_table_structure = True    # detect and parse table structure
pipeline_options.table_structure_options.do_cell_matching = True

ocr_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

print("OCR-enabled converter ready.")
print("Use: result = ocr_converter.convert('path/to/scanned.pdf')")

## 9. Batch Conversion (Multiple Files)

In [ ]:
# Convert multiple documents in one call
sources = [
    "https://arxiv.org/pdf/2408.09869",
    # Add more URLs or local paths here
    # Path("./doc1.pdf"),
    # Path("./doc2.docx"),
]

batch_converter = DocumentConverter()
results = list(batch_converter.convert_all(sources))

print(f"Converted {len(results)} document(s)")
for i, res in enumerate(results):
    md = res.document.export_to_markdown()
    print(f"  Doc {i+1}: {len(md)} chars of markdown")

## 10. Export to HTML

In [ ]:
# Export to HTML
html_output = doc.export_to_html()
Path("output.html").write_text(html_output, encoding="utf-8")

print(f"HTML saved ({len(html_output)} bytes) → output.html")
print("\nHTML preview (first 500 chars):")
print(html_output[:500])

## 11. Chunking for RAG / LLM Pipelines

Split the document into semantically meaningful chunks, ready for embedding.

In [ ]:
from docling.chunking import HybridChunker

chunker = HybridChunker(tokenizer="BAAI/bge-small-en-v1.5")  # or any HF tokenizer
chunks = list(chunker.chunk(dl_doc=doc))

print(f"Total chunks: {len(chunks)}")
print("\n--- First 3 Chunks ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n[Chunk {i+1}] ({len(chunk.text)} chars)")
    print(chunk.text[:300])
    if chunk.meta.headings:
        print(f"  Headings: {chunk.meta.headings}")

## 12. Extract Metadata

In [ ]:
meta = doc.origin if hasattr(doc, "origin") else None
if meta:
    print("Document origin metadata:")
    print(f"  filename  : {meta.filename}")
    print(f"  mimetype  : {meta.mimetype}")
    print(f"  binary hash: {meta.binary_hash}")

if hasattr(doc, "description") and doc.description:
    d = doc.description
    print("\nDocument description:")
    print(f"  title     : {d.title}")
    print(f"  authors   : {d.authors}")
    print(f"  language  : {d.language}")
    print(f"  keywords  : {d.keywords}")

## 13. Image / Figure Extraction

In [ ]:
from docling_core.types.doc import PictureItem
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import DocumentConverter, PdfFormatOption
from IPython.display import display, Image as IPImage
from pathlib import Path

# ── Set your source here (URL or local path) ─────────────────────────────────
SOURCE = source_url          # ← change to Path("your_file.pdf") for a local file

# ── Convert with ALL image options enabled ────────────────────────────────────
pipeline_options = PdfPipelineOptions()
pipeline_options.generate_picture_images = True   # figure-level crops
pipeline_options.generate_page_images    = True   # full-page images (fallback)
pipeline_options.images_scale            = 2.0    # 2× resolution

img_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

result_with_images = img_converter.convert(SOURCE)
doc_with_images    = result_with_images.document

# ── Collect all PictureItems ──────────────────────────────────────────────────
figures = [item for item, _ in doc_with_images.iterate_items() if isinstance(item, PictureItem)]
print(f"Found {len(figures)} figure(s)")

# ── Save every figure to extracted_figures/ ───────────────────────────────────
figures_dir = Path("extracted_figures")
figures_dir.mkdir(exist_ok=True)

saved = 0
for idx, fig in enumerate(figures):
    img_path = figures_dir / f"figure_{idx+1:03d}.png"

    # Approach 1 — direct figure image (works when generate_picture_images=True)
    if fig.image is not None and fig.image.pil_image is not None:
        fig.image.pil_image.save(img_path, format="PNG")
        saved += 1
        continue

    # Approach 2 — crop from the page image using the figure's bounding box
    try:
        prov  = fig.prov[0]                          # provenance: page + bbox
        page  = doc_with_images.pages[prov.page_no - 1]
        if page.image is None or page.image.pil_image is None:
            print(f"Figure {idx+1}: no page image available either, skipping.")
            continue

        page_img = page.image.pil_image
        pw, ph   = page_img.size
        bb       = prov.bbox                         # bbox in doc coordinates

        # docling bbox origin is bottom-left; PIL origin is top-left
        left   = bb.l / page.size.width  * pw
        right  = bb.r / page.size.width  * pw
        top    = (1 - bb.t / page.size.height) * ph
        bottom = (1 - bb.b / page.size.height) * ph
        top, bottom = min(top, bottom), max(top, bottom)

        cropped = page_img.crop((left, top, right, bottom))
        cropped.save(img_path, format="PNG")
        saved += 1
    except Exception as e:
        print(f"Figure {idx+1}: could not extract — {e}")

print(f"\nSaved {saved}/{len(figures)} figures → {figures_dir}/")

# Preview the first 3
for p in sorted(figures_dir.glob("*.png"))[:3]:
    print(p.name)
    display(IPImage(filename=str(p), width=400))

## 14. Markdown with Images

By default `export_to_markdown()` replaces every figure with `<!-- image -->`.  
To get real images you must (a) convert with `generate_picture_images=True` and (b) choose an image mode:

| Mode | Result | File size |
|---|---|---|
| `PLACEHOLDER` (default) | `<!-- image -->` | Small |
| `EMBEDDED` | base64 data-URI inside the `.md` | Large (self-contained) |
| `REFERENCED` | `![](images/picture-1.png)` + separate folder | Moderate |

In [ ]:
from docling_core.types.doc import ImageRefMode

# ── Option A: EMBEDDED (base64 inside the .md — one self-contained file) ──────
md_embedded = doc_with_images.export_to_markdown(image_mode=ImageRefMode.EMBEDDED)
Path("output_with_images_embedded.md").write_text(md_embedded, encoding="utf-8")
print(f"Embedded markdown saved ({len(md_embedded):,} bytes) → output_with_images_embedded.md")

# ── Option B: REFERENCED (images saved as separate PNGs, markdown links them) ─
output_dir = Path("output_referenced")
doc_with_images.save_as_markdown(output_dir, image_mode=ImageRefMode.REFERENCED)
print(f"Referenced markdown saved → {output_dir}/")
print("  Images are in:", list((output_dir).glob("**/*.png"))[:5])

## Summary

| Feature | Method |
|---|---|
| Convert any document | `DocumentConverter().convert(path_or_url)` |
| Export Markdown | `doc.export_to_markdown()` |
| Export JSON | `doc.export_to_dict()` |
| Export HTML | `doc.export_to_html()` |
| Extract tables | `iterate_items()` → filter `TableItem` |
| Extract images | `iterate_items()` → filter `PictureItem` |
| Chunk for RAG | `HybridChunker().chunk(doc)` |
| OCR scanned PDFs | `PdfPipelineOptions(do_ocr=True)` |
| Batch convert | `converter.convert_all([...])` |